# 1. Data

The survey was a Google Form. Participants judged 20 images (AI-generated or human-made; the form scored each answer), then gave their age, gender and weekly hours online. `src/cleaning.py` turns the export into `data/cleaned_dataset.csv`:

- names are replaced by a 12-character SHA-256 hash (`participant_id`)
- the score (`'12 / 20'`) is parsed to `score` and `score_pct`
- free-text hours are parsed: ranges are averaged, per-day answers are multiplied by 7, minutes are converted to hours
- the one unparseable hours answer is imputed with the median

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import Image, Markdown, display

ROOT = Path('..').resolve()
OUT = ROOT / 'outputs'
df = pd.read_csv(ROOT / 'data' / 'cleaned_dataset.csv')

In [ ]:
# Free-text hours answers and how they were parsed (no names shown)
import sys
sys.path.insert(0, str(ROOT / 'src'))
from cleaning import parse_time_hours

raw = pd.read_csv(ROOT / 'data' / 'raw' / 'survey_responses.csv')
hours_col = [c for c in raw.columns if 'hours' in c.lower()][0]
parsed = pd.DataFrame({'answer': raw[hours_col], 'parsed_hours': raw[hours_col].apply(parse_time_hours)})
is_number = pd.to_numeric(parsed['answer'], errors='coerce').notna()
display(Markdown(f'{(~is_number).sum()} of {len(parsed)} answers were not a plain number:'))
display(parsed[~is_number])

In [ ]:
display(Markdown(f'**n = {len(df)}** participants'))
display(df.head())
display(df[['score_pct', 'time_spent_hours', 'age']].describe().round(2))
display(df['gender'].value_counts().rename('n').to_frame())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sns.histplot(df['score_pct'], bins=12, ax=axes[0]).set_title('Score (proportion correct)')
axes[0].axvline(0.5, color='k', linestyle='--', linewidth=1)
sns.histplot(df['time_spent_hours'], bins=20, ax=axes[1]).set_title('Hours online per week')
sns.histplot(data=df, x='age', hue='gender', multiple='stack', bins=15, ax=axes[2]).set_title('Age by gender')
plt.tight_layout()

Notes:
- The dashed line marks chance (50%).
- Most participants are 16–25. Ages over 40 are rare, so age effects at the top of the range rest on few people.
- One participant reported 130 hours a week. `02_analysis` checks whether this answer changes the result.

## Reproduce
All files read here are generated by the pipeline. From the project root:

```bash
python src/run_all.py
```